# Phase 0: mechanistic reproduction of the invariant case

This notebook tests the reduced Phase 0 hypothesis from `RESEARCH_BRIEF.md`:

> A temporal JEPA trained on dynamically immiscible regimes can learn a non-collapsed representation that separates regimes, while an identity-initialized linear predictor acts approximately as the identity on the active latent subspace.

This is **one full baseline run (seed 0)**, not yet a multi-seed reproduction and not an exact replication of every number in the paper. The random-initialization control will be run separately after inspecting this baseline.

In [ ]:
from pathlib import Path
import json
import platform

import numpy as np
import torch
from IPython.display import Image, Markdown, display

from koopman_jepa.config import load_config
from koopman_jepa.phase0 import run

ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists(), "Execute this notebook from the repository root."

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")

## Protocol and decision rule

The run uses the committed smoke configuration at its full 30 epochs: six synthetic regimes, 600 training pairs per regime, an EMA target encoder, and an identity-initialized predictor. CPU is selected for reproducibility.

The current Gate 0 thresholds are:

- effective rank $\geq 2$ (non-collapse);
- held-out linear-probe accuracy $\geq 0.80$;
- active centroid rank $\geq 3$;
- online/target centroid relative error $\leq 0.25$;
- predictor identity error on the active subspace $\leq 0.25$.

In [ ]:
config = load_config(ROOT / "configs" / "phase0_smoke.yaml")
config.model.predictor_init = "identity"
config.train.seed = 0
config.train.epochs = 30
config.train.device = "cpu"
config.to_dict()

## Train and evaluate

The runner writes the complete artifact bundle under `runs/phase0/`. That directory is ignored by Git; the important numerical and visual outputs are embedded below when this notebook is executed.

In [ ]:
run_dir = run(config)
run_dir

## Numerical results

In [ ]:
metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
reported = {
    "effective_rank": metrics["effective_rank"],
    "linear_probe_accuracy": metrics["linear_probe_accuracy"],
    "nearest_centroid_accuracy": metrics["nearest_centroid_accuracy"],
    "kmeans_purity": metrics["kmeans_purity"],
    "active_rank": metrics["active_rank"],
    "active_identity_error": metrics["active_identity_error"],
    "active_eigenvalue_one_error": metrics["active_eigenvalue_one_error"],
    "active_invariance_error": metrics["active_invariance_error"],
    "online_target_centroid_error": metrics["online_target_centroid_error"],
}

table = ["| metric | value |", "|---|---:|"]
for name, value in reported.items():
    rendered = "undefined" if value is None else f"{value:.6f}"
    table.append(f"| `{name}` | {rendered} |")
display(Markdown("\n".join(table)))

In [ ]:
gate_table = ["| Gate 0 check | passed |", "|---|:---:|"]
for name, passed in metrics["gate_checks"].items():
    gate_table.append(f"| `{name}` | {'yes' if passed else 'no'} |")
display(Markdown("\n".join(gate_table)))

active_eigenvalues = [complex(value["real"], value["imag"]) for value in metrics["active_eigenvalues"]]
print("Active-subspace eigenvalues:")
for value in active_eigenvalues:
    print(f"  {value.real:+.6f}{value.imag:+.6f}j")

## Visual diagnostics

In [ ]:
for title, filename in (
    ("Training curves", "loss.png"),
    ("Held-out embeddings (PCA)", "embeddings_pca.png"),
    ("Predictor spectrum", "predictor_spectrum.png"),
):
    display(Markdown(f"### {title}"))
    display(Image(filename=str(run_dir / filename)))

## Interpretation

Passing every check would be evidence that this implementation reproduces the paper's invariant mechanism **qualitatively for one seed**. It would not yet establish robustness. Failing a check is also informative: it identifies whether the issue is collapse, regime identifiability, target alignment, insufficient active rank, or non-identity predictor action.

In [ ]:
if metrics["all_run_checks_passed"]:
    conclusion = (
        "**Baseline result:** all Gate 0 checks passed for seed 0. "
        "Next: run the random-initialization control and then multiple seeds."
    )
else:
    failed = [name for name, passed in metrics["gate_checks"].items() if not passed]
    conclusion = (
        "**Baseline result:** Gate 0 did not pass for seed 0. "
        f"Failed checks: {', '.join(failed)}. "
        "We should diagnose these failures before running a seed sweep."
    )
display(Markdown(conclusion))